In [ ]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

mongo_connector_jar = "/home/provira/Documents/TFM/TFM/src/P2/Explotation Zone/jars/mongo-spark-connector_2.12-3.0.1.jar"
mongo_driver_jar = "/home/provira/Documents/TFM/TFM/src/P2/Explotation Zone/jars/mongo-java-driver-3.12.10.jar"
path_landing = "../../../delta_lake/csv"

builder = SparkSession.builder \
    .appName("MongoDB-Delta Integration") \
    .config("spark.jars", f"{mongo_connector_jar},{mongo_driver_jar}") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.mongodb.read.connection.uri", "mongodb://localhost:27017") \
    .config("spark.mongodb.write.connection.uri", "mongodb://localhost:27017")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

# ✅ Leer desde MongoDB
df = spark.read \
    .format("mongo") \
    .option("uri", "mongodb://localhost:27017") \
    .option("database", "TFM") \
    .option("collection", "tf-idf") \
    .load()

df.printSchema()

df_csv = spark.read.format("delta").load(f"{path_landing}")

In [ ]:
spark.sparkContext._conf.get("spark.jars")

In [ ]:
# Read from MongoDB with proper connection URI
df = spark.read \
    .format("mongo") \
    .option("uri", "mongodb://localhost:27017") \
    .option("database", "TFM") \
    .option("collection", "tf-idf") \
    .load()

df.printSchema()

In [ ]:
from pymongo import MongoClient
client = MongoClient("mongodb://localhost:27017/")
print(client.list_database_names())  # Should show your databases including 'TFM'

In [ ]:
from pyspark.sql import SparkSession

# Option 1: Using packages (recommended)
builder = SparkSession.builder \
    .appName("MongoDB Integration") \
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:3.0.1") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

# Option 2: Using local JAR (if packages don't work)
# builder = SparkSession.builder \
#     .appName("MongoDB Integration") \
#     .config("spark.jars", "/path/to/mongo-spark-connector_2.12-3.0.1.jar") \
#     .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
#     .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = builder.getOrCreate()

try:
    df = spark.read \
        .format("com.mongodb.spark.sql.DefaultSource") \  # Full class name
        .option("uri", "mongodb://localhost:27017/TFM.tf-idf") \  # Full path to collection
        .load()
    
    df.printSchema()
    df.show(5)
except Exception as e:
    print("Error reading from MongoDB:")
    print(e)

In [ ]:
from pyspark.sql import SparkSession

# Create Spark session with MongoDB connector
spark = SparkSession.builder \
    .appName("MongoDBIntegration") \
    .config("spark.mongodb.read.connection.uri", "mongodb://localhost:27017/tfm.tf-idf") \
    .config("spark.mongodb.write.connection.uri", "mongodb://localhost:27017/tfm.tf-idf") \
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:10.2.1") \
    .getOrCreate()

# Now you can read from MongoDB
df = spark.read \
    .format("mongodb") \
    .option("uri", "mongodb://localhost:27017") \
    .option("database", "tfm") \
    .option("collection", "tf-idf") \
    .load()

df.printSchema()


In [ ]:
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession

builder = SparkSession.builder \
    .appName("Delta + MongoDB Integration") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.mongodb.read.connection.uri", "mongodb://localhost:27017/tfm.tf-idf") \
    .config("spark.mongodb.write.connection.uri", "mongodb://localhost:27017/tfm.tf-idf") \
    .config("spark.jars.packages", 
            "org.mongodb.spark:mongo-spark-connector_2.12:10.2.1,io.delta:delta-core_2.12:2.4.0")

spark = configure_spark_with_delta_pip(builder).getOrCreate()


In [ ]:
path_landing = "../../../delta_lake/csv"
path_creation = "/delta_lake/creation"
path_exploitation = "/delta_lake/exploitation"
df_csv = spark.read.format("delta").load(f"{path_landing}")

In [ ]:
df_mongo = spark.read.format("mongodb") \
    .option("database", "tfm") \
    .option("collection", "tf-idf") \
    .load()
